# Taller 5. Arquitectura de Microservicios: Tienda Virtual
Tres microservicios independientes con **FastAPI**: Productos (puerto 8001), Clientes (8002) y Pedidos (8003). Pedidos se comunica con los otros dos por API REST.

Ejecute las celdas en orden (Entorno de ejecución > Ejecutar todas).

## 1. Instalación de dependencias

In [ ]:
%pip install -q fastapi uvicorn httpx requests

## 2. Microservicio de Productos

In [ ]:
%%writefile productos.py
"""Microservicio de Productos: administra el catálogo de la tienda virtual."""
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

app = FastAPI(title="Servicio de Productos", version="1.0")
# CORS: permite que la página web de la tienda (aplicación cliente) consuma la API
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


class ProductoIn(BaseModel):
    nombre: str = Field(min_length=2)
    precio: float = Field(gt=0)
    stock: int = Field(ge=0)


class Producto(ProductoIn):
    id: int


# Lista en memoria (en un sistema real: base de datos propia del servicio)
productos: list[Producto] = [
    Producto(id=1, nombre="Portátil 14 pulgadas", precio=2_800_000, stock=10),
    Producto(id=2, nombre="Mouse inalámbrico", precio=85_000, stock=50),
]


@app.get("/salud")
def salud():
    return {"servicio": "productos", "estado": "activo"}


@app.get("/productos", response_model=list[Producto])
def listar_productos():
    return productos


@app.get("/productos/{producto_id}", response_model=Producto)
def consultar_producto(producto_id: int):
    for p in productos:
        if p.id == producto_id:
            return p
    raise HTTPException(status_code=404, detail="Producto no encontrado")


@app.post("/productos", response_model=Producto, status_code=201)
def registrar_producto(datos: ProductoIn):
    nuevo = Producto(id=max((p.id for p in productos), default=0) + 1, **datos.model_dump())
    productos.append(nuevo)
    return nuevo


## 3. Microservicio de Clientes

In [ ]:
%%writefile clientes.py
"""Microservicio de Clientes: registra y consulta los clientes de la tienda."""
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

app = FastAPI(title="Servicio de Clientes", version="1.0")
# CORS: permite que la página web de la tienda (aplicación cliente) consuma la API
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


class ClienteIn(BaseModel):
    nombre: str = Field(min_length=2)
    correo: str = Field(pattern=r"^[^@\s]+@[^@\s]+\.[^@\s]+$")


class Cliente(ClienteIn):
    id: int


clientes: list[Cliente] = [
    Cliente(id=1, nombre="Laura Gómez", correo="laura.gomez@correo.com"),
]


@app.get("/salud")
def salud():
    return {"servicio": "clientes", "estado": "activo"}


@app.get("/clientes", response_model=list[Cliente])
def listar_clientes():
    return clientes


@app.get("/clientes/{cliente_id}", response_model=Cliente)
def consultar_cliente(cliente_id: int):
    for c in clientes:
        if c.id == cliente_id:
            return c
    raise HTTPException(status_code=404, detail="Cliente no encontrado")


@app.post("/clientes", response_model=Cliente, status_code=201)
def registrar_cliente(datos: ClienteIn):
    if any(c.correo == datos.correo for c in clientes):
        raise HTTPException(status_code=409, detail="El correo ya está registrado")
    nuevo = Cliente(id=max((c.id for c in clientes), default=0) + 1, **datos.model_dump())
    clientes.append(nuevo)
    return nuevo


## 4. Microservicio de Pedidos (depende de Clientes y Productos)

In [ ]:
%%writefile pedidos.py
"""Microservicio de Pedidos: registra pedidos y consulta su estado.

Depende de Clientes y Productos, a los que consulta por HTTP (API REST).
"""
import os
from datetime import datetime

import httpx
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

URL_PRODUCTOS = os.getenv("URL_PRODUCTOS", "http://127.0.0.1:8001")
URL_CLIENTES = os.getenv("URL_CLIENTES", "http://127.0.0.1:8002")
TIEMPO_ESPERA = 2.0   # segundos por intento
REINTENTOS = 2        # intentos adicionales ante fallo de red

app = FastAPI(title="Servicio de Pedidos", version="1.0")
# CORS: permite que la página web de la tienda (aplicación cliente) consuma la API
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


class ItemPedido(BaseModel):
    producto_id: int
    cantidad: int = Field(gt=0)


class PedidoIn(BaseModel):
    cliente_id: int
    items: list[ItemPedido] = Field(min_length=1)


class Pedido(BaseModel):
    id: int
    cliente_id: int
    items: list[ItemPedido]
    total: float
    estado: str
    fecha: str


pedidos: list[Pedido] = []


def consultar_servicio(url: str, nombre_servicio: str) -> httpx.Response:
    """GET con tiempo de espera y reintentos; si el servicio no responde, devuelve 503."""
    ultimo_error = None
    for _ in range(REINTENTOS + 1):
        try:
            return httpx.get(url, timeout=TIEMPO_ESPERA)
        except httpx.RequestError as error:
            ultimo_error = error
    raise HTTPException(
        status_code=503,
        detail=f"Servicio de {nombre_servicio} no disponible ({type(ultimo_error).__name__}). "
               "El pedido no se registró; intente más tarde.",
    )


@app.get("/salud")
def salud():
    return {"servicio": "pedidos", "estado": "activo"}


@app.post("/pedidos", response_model=Pedido, status_code=201)
def registrar_pedido(datos: PedidoIn):
    # 1. Validar el cliente en el servicio de Clientes
    r = consultar_servicio(f"{URL_CLIENTES}/clientes/{datos.cliente_id}", "clientes")
    if r.status_code == 404:
        raise HTTPException(status_code=404, detail="El cliente no existe")

    # 2. Validar cada producto y calcular el total con el servicio de Productos
    total = 0.0
    for item in datos.items:
        r = consultar_servicio(f"{URL_PRODUCTOS}/productos/{item.producto_id}", "productos")
        if r.status_code == 404:
            raise HTTPException(status_code=404, detail=f"El producto {item.producto_id} no existe")
        producto = r.json()
        if producto["stock"] < item.cantidad:
            raise HTTPException(status_code=409, detail=f"Stock insuficiente de {producto['nombre']}")
        total += producto["precio"] * item.cantidad

    pedido = Pedido(
        id=len(pedidos) + 1,
        cliente_id=datos.cliente_id,
        items=datos.items,
        total=total,
        estado="REGISTRADO",
        fecha=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    )
    pedidos.append(pedido)
    return pedido


@app.get("/pedidos/{pedido_id}", response_model=Pedido)
def consultar_estado(pedido_id: int):
    for p in pedidos:
        if p.id == pedido_id:
            return p
    raise HTTPException(status_code=404, detail="Pedido no encontrado")


@app.get("/pedidos", response_model=list[Pedido])
def listar_pedidos():
    return pedidos


## 5. Arranque de los tres servicios
Cada servicio corre en su propio proceso y puerto, como en un despliegue independiente.

In [ ]:
import subprocess, sys, time, requests

procesos = {}
for nombre, puerto in [("productos", 8001), ("clientes", 8002), ("pedidos", 8003)]:
    procesos[nombre] = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", f"{nombre}:app", "--port", str(puerto)],
        stdout=open(f"log_{nombre}.txt", "w"), stderr=subprocess.STDOUT)
time.sleep(4)

for puerto in (8001, 8002, 8003):
    print(requests.get(f"http://127.0.0.1:{puerto}/salud").json())

## 6. Pruebas de las API

In [ ]:
import json
P, C, O = "http://127.0.0.1:8001", "http://127.0.0.1:8002", "http://127.0.0.1:8003"

def mostrar(titulo, r):
    print(f"--- {titulo}: HTTP {r.status_code}")
    print(json.dumps(r.json(), indent=2, ensure_ascii=False), "\n")

In [ ]:
mostrar("GET /productos", requests.get(f"{P}/productos"))
mostrar("POST /productos", requests.post(f"{P}/productos",
        json={"nombre": "Teclado mecánico", "precio": 320000, "stock": 15}))

In [ ]:
mostrar("POST /clientes", requests.post(f"{C}/clientes",
        json={"nombre": "Andrés Pardo", "correo": "andres.pardo@correo.com"}))
mostrar("GET /clientes/2", requests.get(f"{C}/clientes/2"))

### Comunicación entre servicios
Pedidos consulta a Clientes y a Productos para validar y calcular el total (2.800.000 + 2 × 320.000 = 3.440.000).

In [ ]:
mostrar("POST /pedidos", requests.post(f"{O}/pedidos", json={
        "cliente_id": 2,
        "items": [{"producto_id": 1, "cantidad": 1}, {"producto_id": 3, "cantidad": 2}]}))
mostrar("GET /pedidos/1", requests.get(f"{O}/pedidos/1"))

### Validaciones y errores

In [ ]:
mostrar("Cliente inexistente", requests.post(f"{O}/pedidos",
        json={"cliente_id": 99, "items": [{"producto_id": 1, "cantidad": 1}]}))
mostrar("Stock insuficiente", requests.post(f"{O}/pedidos",
        json={"cliente_id": 1, "items": [{"producto_id": 2, "cantidad": 500}]}))
mostrar("Datos inválidos", requests.post(f"{P}/productos",
        json={"nombre": "X", "precio": -5, "stock": 1}))

## 7. Prueba de fallo: se detiene el servicio de Productos
Pedidos debe responder 503 sin bloquearse; Clientes y la consulta de pedidos ya registrados siguen funcionando.

In [ ]:
procesos["productos"].terminate(); procesos["productos"].wait(); time.sleep(1)

mostrar("POST /pedidos con Productos caído", requests.post(f"{O}/pedidos",
        json={"cliente_id": 1, "items": [{"producto_id": 1, "cantidad": 1}]}))
mostrar("GET /clientes/1", requests.get(f"{C}/clientes/1"))
mostrar("GET /pedidos/1", requests.get(f"{O}/pedidos/1"))

## 8. Detener los servicios

In [ ]:
for p in procesos.values():
    p.terminate()
print("Servicios detenidos")